# P3: County Agriculture Merge

Primary merge per `MERGE.md` (§2, **P3**). Combines the five county-grain
agriculture tables into one wide table at **1 row per `county_fips` + `year`**.
Each source is **pivoted wide** first — its categorical descriptors
(commodity/statistic/animal/nutrient/source) become columns — so the final
`county_fips` + `year` join can't fan out.

**Inputs** (all `data/tabular/02_clean/agriculture/...`):
- `crop-yields-clean.csv` — pivoted by `commodity_detail` × `statistic` × `program`
- `livestock-inventory-clean.csv` — **`domain == "TOTAL"` layer only**, pivoted by `commodity_detail` × `statistic` × `program`
- `np-fertilizer-clean.csv` — pivoted by `nutrient` × `source` (kg)
- `np-manure-clean.csv` — pivoted by `animal_category` × `nutrient` (kg)
- `manure-animal-inventory-clean.csv` — pivoted by `animal` × `adjusted` (head)

**Output:** `data/03a_merge_primary/county-agriculture.csv`, one row per county + year.

**Design notes**
- **`program` is kept in the pivoted column name** for crop yields and livestock.
  In census years (2017/2022) both `CENSUS` and `SURVEY` report the same
  `commodity_detail`+`statistic`, which would otherwise collide; naming the two
  columns separately keeps both values instead of silently averaging them.
- **Only the livestock `TOTAL` layer is used.** The herd-size-band rows are the
  *same* operations re-counted into bands (`DATA.md`); including them would risk
  double-counting and explode the column count.
- **Roll-up rows with no `county_fips`** (`OTHER (COMBINED) COUNTIES`) are dropped —
  they have no county to join a station to.
- **Sparse by design.** Crop/livestock run annually 2015–2025; the N&P and
  manure-inventory tables run in ~5-year steps 1950–2017 (overlapping only at
  2017). The outer join is therefore sparse outside that window — expected, per
  `MERGE.md` §6.

In [1]:
import os
import re

import pandas as pd
from functools import reduce

AG = "../../data/tabular/02_clean/agriculture"
OUT_DIR = "../../data/03a_merge_primary"
OUT_FILE = f"{OUT_DIR}/county-agriculture.csv"

KEY = ["county_fips", "year"]


def slug(x):
    """Lowercase, snake_case a categorical label for use in a column name."""
    s = str(x).strip().lower().replace("&", "and").replace("/", "_")
    s = re.sub(r"[(),.]", "", s)
    s = re.sub(r"\s+", "_", s)
    return s


def clean_county_year(df):
    """Standardize the join keys: 5-digit string county_fips (roll-ups dropped),
    integer year."""
    df = df.copy()
    df["county_fips"] = pd.to_numeric(df["county_fips"], errors="coerce")
    df = df.dropna(subset=["county_fips"])
    df["county_fips"] = df["county_fips"].astype(int).astype(str).str.zfill(5)
    df["year"] = df["year"].astype(int)
    return df


def pivot_wide(df, pivot_cols, value_col, prefix, suffix=""):
    """Pivot one long table to 1 row per county_fips+year. The (county, year,
    *pivot_cols) key is asserted unique first so pivot_table never silently
    aggregates. Column names are prefix__<slug>__<slug>...<suffix>."""
    df = clean_county_year(df)
    key = KEY + pivot_cols
    dup = df.duplicated(subset=key).sum()
    assert dup == 0, f"{prefix}: {dup} duplicate rows on {key} — pivot would aggregate"
    wide = df.pivot_table(index=KEY, columns=pivot_cols, values=value_col, aggfunc="mean")
    if isinstance(wide.columns, pd.MultiIndex):
        wide.columns = [prefix + "__" + "__".join(slug(p) for p in tup) + suffix for tup in wide.columns]
    else:
        wide.columns = [prefix + "__" + slug(c) + suffix for c in wide.columns]
    wide = wide.reset_index()
    print(f"{prefix}: {wide.shape[0]:,} county-years x {wide.shape[1] - 2} value cols")
    return wide

## Step 1: Crop yields

Pivoted by `commodity_detail` × `statistic` × `program`. Units vary by column
(BU / TONS / BU-per-ACRE) but are a deterministic function of
`commodity_detail`+`statistic`, so they're documented in `DATA.md` rather than
embedded in every column name.

In [2]:
crop = pd.read_csv(f"{AG}/crop-yields-clean.csv")
crop_wide = pivot_wide(
    crop,
    pivot_cols=["commodity_detail", "statistic", "program"],
    value_col="value",
    prefix="cropyield",
)
crop_wide.head(3)

cropyield: 1,025 county-years x 28 value cols


,county_fips,year,cropyield__barley__production__census,cropyield__corn__acres_planted__survey,cropyield__corn_grain__production__census,cropyield__corn_grain__production__survey,cropyield__corn_grain__yield__survey,cropyield__corn_silage__production__census,cropyield__corn_silage__production__survey,cropyield__corn_silage__yield__survey,...,cropyield__oats__production__survey,cropyield__oats__yield__survey,cropyield__rye__production__census,cropyield__soybeans__acres_planted__survey,cropyield__soybeans__production__census,cropyield__soybeans__production__survey,cropyield__soybeans__yield__survey,cropyield__wheat__production__census,cropyield__wheat_spring_excl_durum__production__census,cropyield__wheat_winter__production__census
0,19001,2015,NaN,112000.0,NaN,18954000.0,176.5,NaN,NaN,NaN,...,NaN,NaN,NaN,106500.0,NaN,5849000.0,55.1,NaN,NaN,NaN
1,19001,2016,NaN,115000.0,NaN,21659000.0,190.3,NaN,NaN,NaN,...,NaN,NaN,NaN,107000.0,NaN,6332000.0,59.3,NaN,NaN,NaN
2,19001,2017,NaN,108500.0,15907191.0,18235000.0,175.2,37037.0,NaN,NaN,...,NaN,NaN,NaN,110500.0,5410804.0,5554000.0,50.5,NaN,NaN,NaN


## Step 2: Livestock inventory (TOTAL layer)

Restricted to `domain == "TOTAL"`, then pivoted by
`commodity_detail` × `statistic` × `program`. `INVENTORY` values are head counts,
`OPERATIONS WITH INVENTORY` are farm counts.

In [3]:
livestock = pd.read_csv(f"{AG}/livestock-inventory-clean.csv")
livestock_total = livestock[livestock["domain"] == "TOTAL"]
print(f"TOTAL layer: {len(livestock_total):,} of {len(livestock):,} rows")
livestock_wide = pivot_wide(
    livestock_total,
    pivot_cols=["commodity_detail", "statistic", "program"],
    value_col="value",
    prefix="livestock",
)
livestock_wide.head(3)

TOTAL layer: 7,646 of 21,868 rows
livestock: 1,089 county-years x 29 value cols


,county_fips,year,livestock__cattle_excl_cows__inventory__census,livestock__cattle_excl_cows__operations_with_inventory__census,livestock__cattle_cows__inventory__census,livestock__cattle_cows__operations_with_inventory__census,livestock__cattle_cows_beef__inventory__census,livestock__cattle_cows_beef__inventory__survey,livestock__cattle_cows_beef__operations_with_inventory__census,livestock__cattle_cows_milk__inventory__census,...,livestock__goats_meat_and_other__inventory__census,livestock__goats_meat_and_other__operations_with_inventory__census,livestock__goats_milk__inventory__census,livestock__goats_milk__operations_with_inventory__census,livestock__hogs__inventory__census,livestock__hogs__operations_with_inventory__census,livestock__sheep_incl_lambs__inventory__census,livestock__sheep_incl_lambs__operations_with_inventory__census,livestock__sheep_incl_lambs_hair_sheep_or_wool-hair_crosses__inventory__census,livestock__sheep_incl_lambs_hair_sheep_or_wool-hair_crosses__operations_with_inventory__census
0,19001,2015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,19001,2016,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,19001,2017,27120.0,310.0,20236.0,300.0,NaN,NaN,300.0,NaN,...,NaN,1.0,NaN,1.0,52615.0,19.0,1267.0,33.0,NaN,NaN


## Step 3: N & P from fertilizer

Pivoted by `nutrient` (N/P) × `source` (farm/nonfarm/total), kilograms.

In [4]:
npfert = pd.read_csv(f"{AG}/np-fertilizer-clean.csv")
npfert_wide = pivot_wide(
    npfert,
    pivot_cols=["nutrient", "source"],
    value_col="value_kg",
    prefix="npfert",
    suffix="_kg",
)
npfert_wide.head(3)

npfert: 1,485 county-years x 6 value cols


,county_fips,year,npfert__n__farm_kg,npfert__n__nonfarm_kg,npfert__n__total_kg,npfert__p__farm_kg,npfert__p__nonfarm_kg,npfert__p__total_kg
0,19001,1950,NaN,NaN,92656.898438,NaN,NaN,156006.09375
1,19001,1954,NaN,NaN,512755.406250,NaN,NaN,301432.50000
2,19001,1959,NaN,NaN,631447.812500,NaN,NaN,345982.90625


## Step 4: N & P from manure

Pivoted by `animal_category` × `nutrient`, kilograms.

In [5]:
npmanure = pd.read_csv(f"{AG}/np-manure-clean.csv")
npmanure_wide = pivot_wide(
    npmanure,
    pivot_cols=["animal_category", "nutrient"],
    value_col="value_kg",
    prefix="npmanure",
    suffix="_kg",
)
npmanure_wide.head(3)

npmanure: 1,485 county-years x 10 value cols


,county_fips,year,npmanure__cattle__n_kg,npmanure__cattle__p_kg,npmanure__hogs__n_kg,npmanure__hogs__p_kg,npmanure__other__n_kg,npmanure__other__p_kg,npmanure__poultry__n_kg,npmanure__poultry__p_kg,npmanure__total__n_kg,npmanure__total__p_kg
0,19001,1950,2.208110e+06,563783.531235,9.186584e+05,408292.623720,249859.3119,43347.721200,138167.711874,55182.655677,3.514795e+06,1.070607e+06
1,19001,1954,2.880253e+06,790693.997260,1.151496e+06,511775.826480,103682.6154,18031.759200,153234.332277,61232.669577,4.288665e+06,1.381734e+06
2,19001,1959,2.984105e+06,827457.583109,1.372703e+06,610090.251937,177982.7600,30925.402609,134007.713642,53602.503254,4.668799e+06,1.522076e+06


## Step 5: Manure animal inventory

Pivoted by `animal` × `adjusted`, head counts. The `adjusted` flag
(`adj` = USDA slaughter-weight-adjusted, `raw` = unadjusted) is kept as part of
the column name so both series survive.

In [6]:
manure_inv = pd.read_csv(f"{AG}/manure-animal-inventory-clean.csv")
manure_inv["adjusted"] = manure_inv["adjusted"].map({True: "adj", False: "raw"})
manure_inv_wide = pivot_wide(
    manure_inv,
    pivot_cols=["animal", "adjusted"],
    value_col="head_count",
    prefix="manureinv",
    suffix="_head",
)
manure_inv_wide.head(3)

manureinv: 1,485 county-years x 10 value cols


,county_fips,year,manureinv__all_cattle_and_calves__adj_head,manureinv__beef_cows__adj_head,manureinv__broilers__adj_head,manureinv__hogs_and_pigs__adj_head,manureinv__horses_and_ponies__adj_head,manureinv__layers__adj_head,manureinv__milk_cows__adj_head,manureinv__other_cattle_and_calves__raw_head,manureinv__sheep_and_lambs__adj_head,manureinv__turkeys__adj_head
0,19001,1950,52030.0,11894.0,2240.0,98957.0,3342.0,240059.0,9498.0,30638.0,14499.0,2436.0
1,19001,1954,68819.0,20565.0,2240.0,124038.0,0.0,268142.0,7958.0,40296.0,15834.0,28.0
2,19001,1959,72433.0,20747.0,0.0,146988.0,886.0,235408.0,6233.0,45453.0,20839.0,60.0


## Step 6: Outer-merge on `county_fips` + `year`, then save

Each input is already 1 row per county-year, so an outer join on
`county_fips` + `year` unions the (county, year) coverage without any fan-out.

In [7]:
frames = [crop_wide, livestock_wide, npfert_wide, npmanure_wide, manure_inv_wide]
df = reduce(lambda l, r: l.merge(r, on=KEY, how="outer"), frames)

assert not df.duplicated(subset=KEY).any(), "Output grain violated: duplicate (county_fips, year) rows"

df = df.sort_values(KEY).reset_index(drop=True)

print(f"Final shape: {df.shape}")
print(f"County-years: {len(df):,}  |  distinct counties: {df['county_fips'].nunique()}  |  "
      f"year range: {df['year'].min()}–{df['year'].max()}")

def block_coverage(prefix):
    cols = [c for c in df.columns if c.startswith(prefix + "__")]
    return df[cols].notna().any(axis=1).sum()

print("\nCounty-years with any data, by source block:")
for pfx in ["cropyield", "livestock", "npfert", "npmanure", "manureinv"]:
    print(f"  {pfx:11s}: {block_coverage(pfx):>5,}")
overlap = df[(df['year'] >= 2015) & (df['year'] <= 2017)]
print(f"Rows in the 2015–2017 crop/N&P overlap window: {len(overlap):,}")

Final shape: (2475, 85)
County-years: 2,475  |  distinct counties: 99  |  year range: 1950–2025

County-years with any data, by source block:
  cropyield  : 1,025
  livestock  : 1,089
  npfert     : 1,485
  npmanure   : 1,485
  manureinv  : 1,485
Rows in the 2015–2017 crop/N&P overlap window: 297


In [8]:
os.makedirs(OUT_DIR, exist_ok=True)
df.to_csv(OUT_FILE, index=False)
print(f"Saved {len(df):,} rows x {df.shape[1]} cols -> {OUT_FILE}")

Saved 2,475 rows x 85 cols -> ../../data/03a_merge_primary/county-agriculture.csv
